# MongoDB | Assignment
**Course:** PW Skills — Data Science / Machine Learning

This notebook covers 23 theoretical questions and 11 practical (PyMongo) questions on MongoDB, using the Superstore dataset.

> **Setup note:** The practical section uses `pymongo` to connect to a MongoDB instance. To run these cells yourself:
> ```bash
> pip install pymongo pandas
> ```
> and have a MongoDB server running locally (default `mongodb://localhost:27017/`) or a MongoDB Atlas connection string. This sandbox environment has no MongoDB server or internet access, so a **synthetic Superstore-style dataset** (2,000 rows, same schema as the real Kaggle Superstore dataset) is generated below and the query logic is verified against it using `pandas` as ground truth — the PyMongo code shown is the correct, standard way to perform each operation against a real MongoDB collection, and the printed outputs are the actual/expected result values.


# Part 1 — Theoretical Questions

### 1. What are the key differences between SQL and NoSQL databases?

| Aspect | SQL (Relational) | NoSQL (e.g., MongoDB) |
|---|---|---|
| **Data model** | Tables with fixed rows/columns | Flexible — documents (JSON/BSON), key-value, column-family, or graph |
| **Schema** | Rigid, predefined schema | Dynamic/schema-less (or schema validation is optional) |
| **Relationships** | Enforced via foreign keys and joins | Typically modeled via embedding or references; joins are limited (`$lookup`) |
| **Scaling** | Primarily **vertical** (bigger server) | Primarily **horizontal** (sharding across many servers) |
| **Consistency** | Strong (ACID) by default | Historically eventual consistency; modern MongoDB supports multi-document ACID transactions too |
| **Query language** | SQL (standardized) | Database-specific query APIs (e.g., MongoDB Query Language / aggregation pipeline) |
| **Best suited for** | Structured data with complex relationships (banking, ERP) | Unstructured/semi-structured, rapidly evolving, high-volume data (content management, real-time analytics, IoT) |

In short, SQL databases prioritize structure and strong consistency guarantees, while NoSQL databases prioritize flexibility and horizontal scalability, often at the cost of some rigid consistency guarantees (though this gap has narrowed significantly in modern NoSQL systems like MongoDB).


### 2. What makes MongoDB a good choice for modern applications?

- **Flexible, document-based schema:** Data is stored as JSON-like BSON documents, which map naturally to objects in application code (no complex ORM impedance mismatch), and the schema can evolve without costly migrations.
- **Horizontal scalability:** Built-in **sharding** allows MongoDB to scale out across many servers to handle very large datasets and high throughput.
- **High availability:** **Replica sets** provide automatic failover and data redundancy.
- **Rich query capabilities:** Supports powerful queries, indexing (including geospatial, text, and compound indexes), and a flexible **aggregation pipeline** for complex data transformations/analytics.
- **Developer productivity:** Its document model closely mirrors how developers think in code (nested objects/arrays), reducing development time for many modern application patterns.
- **Multi-cloud, managed offering (MongoDB Atlas):** Simplifies deployment, scaling, monitoring, and backups.
- **Good fit for agile development:** Since the schema isn't rigid, it's well suited to applications with rapidly evolving requirements (startups, MVPs, content-heavy apps).

These characteristics make MongoDB popular for modern web/mobile applications, real-time analytics, content management, IoT, and microservices architectures where flexibility and scale matter more than rigid relational structure.


### 3. Explain the concept of collections in MongoDB.

A **collection** in MongoDB is the equivalent of a "table" in a relational database — it's a **grouping of documents**, typically documents that are similar in purpose (e.g., an `orders` collection, a `users` collection).

Key characteristics:
- Collections are **schema-less** by default — documents within the same collection don't need to have identical fields or field types (though schema *validation* rules can optionally be applied).
- Each document within a collection is uniquely identified by an `_id` field (auto-generated as an `ObjectId` if not provided).
- Collections exist within a **database**, and a MongoDB server can host multiple databases, each containing multiple collections.
- Collections are created implicitly the first time a document is inserted into them (no explicit `CREATE TABLE`-style step is required, though they can also be created explicitly with options like validation rules or capped size).


### 4. How does MongoDB ensure high availability using replication?

MongoDB achieves high availability through **Replica Sets** — a group of `mongod` processes that maintain the same data set.

**How it works:**
- A replica set consists of one **Primary** node (which receives all write operations) and multiple **Secondary** nodes (which replicate data from the primary via the **oplog**, an operations log).
- If the primary node becomes unavailable (crash, network partition, maintenance), the remaining nodes in the replica set automatically hold an **election** and promote one of the secondaries to become the new primary — this is called **automatic failover**, typically completing within seconds.
- Secondaries can also serve **read operations** (if the application opts into reading from secondaries), which can help distribute read load.
- Because data is continuously replicated to multiple nodes, the loss of any single node (or even multiple, as long as a majority remains) does not result in data loss, providing both **redundancy** and **fault tolerance**.
- Replica sets are usually deployed with an odd number of members (e.g., 3 or 5) to ensure elections always have a clear majority.


### 5. What are the main benefits of MongoDB Atlas?

**MongoDB Atlas** is MongoDB's official fully-managed, cloud-hosted Database-as-a-Service (DBaaS) offering. Its main benefits:

- **Fully managed infrastructure:** Automates provisioning, patching, backups, and monitoring — no need to manually manage servers.
- **Multi-cloud support:** Deployable on AWS, Google Cloud, and Azure, with the ability to even span multiple clouds/regions for a single cluster.
- **Automatic scaling:** Supports both vertical (bigger instances) and horizontal (auto-sharding) scaling, including "auto-scale" options that adjust capacity based on load.
- **Built-in high availability:** Replica sets and automated failover are configured out-of-the-box.
- **Security features:** Encryption at rest and in transit, IP whitelisting, VPC peering, fine-grained role-based access control (RBAC), and integration with enterprise identity providers.
- **Integrated tooling:** Built-in performance monitoring/alerting, a visual query/aggregation builder, full-text search (Atlas Search), data visualization (Atlas Charts), and serverless options.
- **Global clusters:** Enables geographically distributed deployments for low-latency access close to users worldwide.
- **Free tier:** Offers a free-forever shared cluster tier, making it easy to prototype and learn without upfront infrastructure cost.


### 6. What is the role of indexes in MongoDB, and how do they improve performance?

An **index** in MongoDB is a special data structure (typically a B-tree) that stores a small, efficiently searchable portion of a collection's data — specifically, the values of one or more fields, ordered in a way that allows fast lookups.

**Role/benefits:**
- **Without an index**, MongoDB must perform a **collection scan** — examining every document in the collection to find matches for a query — which is slow for large collections.
- **With an appropriate index**, MongoDB can navigate directly to the relevant documents, dramatically reducing the number of documents scanned and improving query response time.
- Indexes also support efficient **sorting** (avoiding an expensive in-memory sort) and can enforce constraints (e.g., a unique index prevents duplicate values in a field).

**Common index types:** single-field, compound (multiple fields), multikey (for array fields), text (for full-text search), geospatial (for location queries), and hashed (for even sharding distribution).

**Trade-off:** Indexes speed up reads but add overhead to writes (since every insert/update/delete must also update relevant indexes) and consume additional storage — so indexes should be created deliberately, based on actual query patterns, not indiscriminately.


### 7. Describe the stages of the MongoDB aggregation pipeline.

The **Aggregation Pipeline** processes documents through a sequence of **stages**, where each stage transforms the documents and passes the result to the next stage — conceptually similar to a Unix pipeline. Common stages include:

- **`$match`** — filters documents (similar to a `WHERE` clause in SQL), passing through only documents that meet specified criteria.
- **`$group`** — groups documents by a specified key and computes aggregate values (sum, average, count, min, max, etc.) per group — similar to SQL's `GROUP BY`.
- **`$project`** — reshapes documents: includes/excludes fields, renames fields, or computes new derived fields.
- **`$sort`** — orders documents by one or more fields.
- **`$limit`** / **`$skip`** — restrict the number of documents returned / skip a number of documents (useful for pagination).
- **`$lookup`** — performs a left-outer-join-like operation with another collection.
- **`$unwind`** — deconstructs an array field, producing one output document per array element.
- **`$addFields`** — adds new computed fields to documents without removing existing ones.

Stages are combined in an ordered array (a "pipeline"), e.g., `[{"$match": ...}, {"$group": ...}, {"$sort": ...}]`, and are the primary mechanism for performing complex analytics and data transformations directly within MongoDB.


### 8. What is sharding in MongoDB? How does it differ from replication?

**Sharding** is MongoDB's method for **horizontal scaling** — distributing (partitioning) a large dataset across multiple servers ("shards"), where each shard holds only a subset of the total data.

**Key components:**
- **Shard key** — a field (or fields) used to determine how documents are distributed across shards.
- **Shards** — each shard is typically itself a replica set, storing a portion of the sharded data.
- **`mongos`** — a routing process that directs client queries to the appropriate shard(s).
- **Config servers** — store metadata about the cluster's shard/chunk distribution.

**Sharding vs. Replication:**

| Aspect | Sharding | Replication |
|---|---|---|
| Purpose | Distribute data across servers to scale **storage & throughput** | Duplicate the *same* data across servers for **redundancy & availability** |
| Data on each node | **Different** subset of the data on each shard | **Identical** copy of the full data set on each replica |
| Solves | "Too much data / too many operations for one server" | "What if one server fails?" |
| Typical combination | Shards are usually *also* replica sets — the two techniques are complementary, not mutually exclusive | |

In short: sharding scales out data volume/throughput by splitting data across machines, while replication provides redundancy by duplicating the same data across machines — production MongoDB deployments commonly use **both together**.


### 9. What is PyMongo, and why is it used?

**PyMongo** is the official Python driver for MongoDB — a library that allows Python applications to connect to, query, and manipulate data in a MongoDB database.

**Why it's used:**
- Provides a Pythonic API for MongoDB operations: `insert_one()`/`insert_many()`, `find()`, `update_one()`/`update_many()`, `delete_one()`/`delete_many()`, and the aggregation pipeline (`aggregate()`).
- Handles the underlying wire protocol communication with MongoDB (connection pooling, BSON serialization/deserialization) transparently.
- Documents are represented as native Python dictionaries, and query results (BSON) are automatically converted to Python objects, making integration with the rest of a Python data/application stack seamless.
- Widely used in data science and backend development workflows for loading, transforming, and querying document data stored in MongoDB (e.g., in ETL pipelines, Flask/Django backends, or notebooks like this one).
- Installed via `pip install pymongo`.


### 10. What are the ACID properties in the context of MongoDB transactions?

**ACID** stands for **Atomicity, Consistency, Isolation, Durability** — properties that guarantee reliable processing of database transactions.

- **Atomicity:** All operations within a transaction either succeed together or fail together (all-or-nothing) — there's no partial application of a transaction's changes.
- **Consistency:** A transaction brings the database from one valid state to another, respecting all defined rules/constraints (e.g., schema validation, unique indexes).
- **Isolation:** Concurrent transactions don't interfere with each other — the intermediate state of one transaction is invisible to others until it commits.
- **Durability:** Once a transaction is committed, its changes are permanently saved, even in the event of a system crash (backed by the write-ahead journal).

**In MongoDB specifically:**
- **Single-document operations** have always been atomic in MongoDB, even before multi-document transactions existed, because a document (including nested arrays/sub-documents) is the atomic unit of storage.
- Since **MongoDB 4.0** (for replica sets) and **4.2** (for sharded clusters), MongoDB also supports **multi-document ACID transactions**, allowing multiple operations across multiple documents/collections to be grouped into a single all-or-nothing transaction — using a session (`client.start_session()`) with `with_transaction()` or manual `start_transaction()`/`commit_transaction()` calls.


### 11. What is the purpose of MongoDB's `explain()` function?

The **`explain()`** function/method returns detailed information about how MongoDB **executes a given query** — it's MongoDB's equivalent of SQL's `EXPLAIN`/query execution plan.

**What it reveals:**
- Whether the query used an **index** ("IXSCAN") or performed a full **collection scan** ("COLLSCAN") — the latter being much slower on large collections.
- The number of documents **examined** vs. the number **returned** — a large gap suggests an inefficient query/missing index.
- Execution time and the overall **query plan** chosen by MongoDB's query optimizer, including any rejected alternative plans.
- Different verbosity modes: `"queryPlanner"` (the plan MongoDB would use), `"executionStats"` (actual runtime statistics after executing the query), and `"allPlansExecution"` (stats for all candidate plans considered).

**Why it's useful:** It's the primary diagnostic tool for **query performance tuning** — helping developers decide where indexes are needed and verifying that existing indexes are actually being used as intended. Example usage: `db.orders.find({"Region": "West"}).explain("executionStats")`.


### 12. How does MongoDB handle schema validation?

Although MongoDB is fundamentally schema-less/flexible, it supports **optional schema validation** to enforce structure and data-quality rules when desired, using **JSON Schema** validation rules attached to a collection.

**How it works:**
- Validation rules are defined when creating a collection (`db.createCollection()`) or added later (`collMod` command), using the `validator` option with a `$jsonSchema` (or query-style) expression.
- Rules can specify required fields, allowed data types (`bsonType`), value ranges/patterns, enumerated allowed values, and more.
- The `validationLevel` option controls how strictly rules are enforced: `"strict"` (validate all inserts/updates) or `"moderate"` (only validate updates to documents that already satisfy the rules, leaving old non-conforming documents alone).
- The `validationAction` option controls what happens on a violation: `"error"` (reject the write) or `"warn"` (log a warning but still allow the write).

This gives MongoDB a flexible middle ground: teams can start schema-less for rapid prototyping, then progressively add validation rules as the application's data model stabilizes, without a rigid, all-or-nothing schema like a traditional RDBMS.


### 13. What is the difference between a Primary and a Secondary node in a replica set?

| Aspect | Primary Node | Secondary Node |
|---|---|---|
| **Writes** | Receives **all write operations** for the replica set | Cannot accept writes directly (read-only, unless promoted) |
| **Reads** | Handles reads by default | Can optionally serve reads if the client opts in via read preference (e.g., for read-scaling or reducing load on the primary) |
| **Data replication** | Source of truth — changes are logged to the **oplog** | Continuously replicates data from the primary's oplog to stay in sync |
| **Role changes** | Can be voted out and demoted if it becomes unreachable or loses an election | Can be automatically elected/promoted to primary if the current primary becomes unavailable |
| **Count per replica set** | Exactly **one** at any given time | **One or more** (a typical replica set has 2+ secondaries) |

In short, the Primary is the single node authoritative for writes at any moment, while Secondaries maintain synchronized copies of the data and stand ready to take over as Primary through an automatic election if needed.


### 14. What security mechanisms does MongoDB provide for data protection?

MongoDB offers several layers of security controls:

- **Authentication:** Verifies user identity before granting database access — supports SCRAM (username/password), x.509 certificates, LDAP, and Kerberos (in Enterprise/Atlas editions).
- **Authorization (Role-Based Access Control - RBAC):** Assigns users specific **roles** (e.g., `read`, `readWrite`, `dbAdmin`) scoped to specific databases/collections, following the principle of least privilege.
- **Encryption in transit:** TLS/SSL encrypts data moving between the client and the MongoDB server.
- **Encryption at rest:** Encrypts data files on disk (native encrypted storage engine in Enterprise, or via file-system/OS-level encryption; Atlas encrypts at rest by default).
- **Field-Level Encryption (Client-Side Field Level Encryption):** Encrypts specific sensitive fields on the client side before they're even sent to the server, so the server never sees the plaintext.
- **Network security:** IP whitelisting/allowlists, VPC/private network peering, and firewall rules to restrict which hosts can connect.
- **Auditing:** Enterprise/Atlas editions can log detailed system activity (who accessed what, and when) for compliance purposes.
- **Disabling unnecessary interfaces:** Best practice guidance includes binding MongoDB to specific IPs (not `0.0.0.0` by default) and disabling anonymous/unauthenticated access in production.


### 15. Explain the concept of embedded documents and when they should be used.

**Embedded documents** refer to nesting one document (or an array of documents) directly *inside* another document, rather than storing related data in a separate collection and linking via a reference (foreign key).

**Example:** an `orders` document might embed the shipping address directly:
```json
{
  "order_id": "US-2023-1001",
  "customer": "Jane Doe",
  "shipping_address": { "city": "Los Angeles", "state": "CA", "zip": "90001" }
}
```

**When to use embedding (vs. referencing a separate collection):**
- When the embedded data is almost always accessed **together** with the parent document (e.g., an order and its shipping address) — embedding avoids the need for a separate query/join.
- When there's a clear **"contains" / one-to-few** relationship, and the embedded data doesn't need to be queried independently or shared across multiple parent documents.
- When you want **atomic updates** — since a single document (including its embedded sub-documents) is updated atomically in MongoDB, embedding related data that must stay consistent together is often preferable.

**When to prefer referencing instead:** when the "child" data is large, grows unboundedly (e.g., thousands of comments on a post), is shared/reused across many parent documents, or needs to be queried/updated independently of its parent — referencing (storing an ID and doing a separate lookup or `$lookup`) avoids documents growing too large and avoids data duplication.


### 16. What is the purpose of MongoDB's `$lookup` stage in aggregation?

The **`$lookup`** stage performs a **left outer join**-style operation between the current collection and another ("foreign") collection within the aggregation pipeline — something MongoDB otherwise doesn't support natively, since it's a document database rather than a relational one.

**Syntax (basic form):**
```python
{
  "$lookup": {
    "from": "customers",          # the foreign collection
    "localField": "customer_id",  # field in the current collection
    "foreignField": "_id",        # field in the foreign collection
    "as": "customer_info"         # name of the new array field holding matched documents
  }
}
```

**Purpose:**
- Allows combining related data that is stored across **separate collections** (a common pattern when data was intentionally normalized/referenced rather than embedded) into a single result set for querying/reporting purposes.
- The matched documents from the foreign collection are added as an **array field** to each document from the current collection (documents with no match get an empty array).
- Frequently paired with `$unwind` afterward to "flatten" the resulting array when a one-to-one relationship is expected.

This is MongoDB's mechanism for achieving SQL-`JOIN`-like functionality when data has been split across multiple collections rather than embedded.


### 17. What are some common use cases for MongoDB?

- **Content management systems (CMS):** flexible, evolving content schemas (articles, media metadata) suit MongoDB's document model well.
- **Real-time analytics & dashboards:** the aggregation pipeline enables complex analytical queries directly within the database.
- **Catalogs and product data (e-commerce):** products often have highly variable attributes (a shirt has size/color, a laptop has RAM/CPU) — embedding these as flexible sub-documents avoids the sparse-columns problem of a rigid relational schema.
- **IoT and sensor/time-series data:** high write throughput and (via time-series collections) efficient storage for time-stamped data streams.
- **Mobile and single-view applications:** data that maps naturally to JSON objects consumed directly by apps/APIs.
- **User profiles and personalization data:** flexible, evolving fields per user without constant schema migrations.
- **Log and event data storage:** high-volume, semi-structured event/log ingestion.
- **Gaming (leaderboards, player state, session data):** requires fast reads/writes at scale with flexible data shapes.


### 18. What are the advantages of using MongoDB for horizontal scaling?

- **Native sharding support:** MongoDB was designed from the ground up with horizontal scalability in mind, via automatic, built-in sharding — data is automatically partitioned and balanced across shards as the dataset grows.
- **Automatic load balancing:** MongoDB's balancer automatically migrates "chunks" of data between shards to keep the cluster evenly distributed as data grows or access patterns change.
- **Linear scalability for reads & writes:** Adding more shards increases both storage capacity and read/write throughput roughly proportionally (assuming a well-chosen shard key that avoids "hot spots").
- **Transparent to the application:** The `mongos` query router abstracts away the sharded topology — client applications interact with the cluster the same way they would with a single MongoDB instance.
- **Combines with replication:** Each shard is typically also a replica set, so horizontal scaling doesn't come at the expense of high availability.
- **Flexible shard key strategies:** Supports ranged, hashed, and (in newer versions) zone-based sharding to suit different data-distribution and query patterns.

This makes MongoDB well suited for applications with data volumes or throughput requirements that exceed what a single, vertically-scaled server could handle cost-effectively.


### 19. How do MongoDB transactions differ from SQL transactions?

| Aspect | SQL (RDBMS) Transactions | MongoDB Transactions |
|---|---|---|
| **Scope of atomicity by default** | Every multi-statement transaction requires explicit `BEGIN`/`COMMIT` to be atomic | **Single-document** writes are *always* atomic by default, even without an explicit transaction — because a document (with embedded data) is often already the natural transactional unit |
| **Multi-document/table transactions** | Core, foundational feature since inception | Added later (MongoDB 4.0 for replica sets, 4.2 for sharded clusters) as an *additional* capability for cases embedding can't cover |
| **Typical need for use** | Very common — relational schemas frequently spread related data across many tables, requiring transactions to keep them consistent | Less common in practice — because embedding is preferred for tightly related data, many MongoDB use cases avoid needing multi-document transactions at all |
| **Performance overhead** | Well-optimized, deeply embedded in RDBMS engines | Multi-document transactions carry more relative overhead in MongoDB and are recommended only when truly necessary (not as the default pattern) |
| **API** | `BEGIN TRANSACTION` / `COMMIT` / `ROLLBACK` (SQL statements) | `session.start_transaction()` / `session.commit_transaction()` / `session.abort_transaction()` (via a client session object) |

**In short:** SQL databases are built around the assumption that transactions spanning multiple tables are the norm, while MongoDB's document model is designed to minimize the *need* for multi-document transactions by embedding related data together — but full ACID multi-document transactions are available as a fallback for the cases where they're genuinely required (e.g., transferring funds between two separate account documents).


### 20. What are the main differences between capped collections and regular collections?

| Aspect | Capped Collection | Regular Collection |
|---|---|---|
| **Size** | **Fixed size** (set at creation, in bytes and/or max document count) | Grows dynamically, no fixed upper bound |
| **Insertion order** | Documents are stored strictly in **insertion order** | No guaranteed physical storage order |
| **Overwrite behavior** | Once full, **automatically overwrites the oldest documents** to make room for new ones (FIFO behavior) — like a circular buffer | Never automatically deletes documents; documents persist until explicitly deleted |
| **Document updates** | Updates that would **increase document size are not allowed** (would break the fixed-size guarantee) | Documents can be freely updated, including growing in size |
| **Deletion** | Individual document deletion is not typically supported (deletion happens automatically via the FIFO cap) | Full support for arbitrary `delete_one()`/`delete_many()` |
| **Performance** | Very fast inserts and retrieval in insertion order (no index needed for that ordering) | Standard performance, relies on indexes for fast lookups |
| **Typical use cases** | Logging, caching, storing the most recent N events (e.g., real-time chat message buffers, monitoring/audit logs) | General-purpose application data storage |

Capped collections trade flexibility for very high-performance, bounded-size, insertion-ordered storage — ideal when you only ever care about "the most recent N records" and want old data to automatically roll off.


### 21. What is the purpose of the `$match` stage in MongoDB's aggregation pipeline?

The **`$match`** stage **filters documents**, passing along to the next pipeline stage only the documents that satisfy specified query criteria — it is the aggregation-pipeline equivalent of a `find()` filter (or SQL's `WHERE` clause).

**Purpose and best practices:**
- Used to narrow down the working set of documents *before* more expensive operations (like `$group` or `$lookup`) are applied, which improves performance by reducing the number of documents processed downstream.
- **Best practice:** place `$match` stages as **early as possible** in the pipeline (ideally as the very first stage) — this allows MongoDB to use indexes to satisfy the match efficiently, similar to how a `WHERE` clause benefits from an index in SQL.
- Uses the same query operators available in `find()` — e.g., `$eq`, `$gt`, `$lt`, `$in`, `$regex`, `$and`, `$or`.

**Example:**
```python
{"$match": {"Region": "West", "Sales": {"$gt": 500}}}
```
This keeps only documents where `Region` is `"West"` and `Sales` is greater than 500, discarding everything else before further pipeline processing.


### 22. How can you secure access to a MongoDB database?

- **Enable authentication:** Require all clients to authenticate (username/password via SCRAM, x.509 certificates, LDAP, or Kerberos) — never run a production MongoDB instance with authentication disabled.
- **Use Role-Based Access Control (RBAC):** Grant each application/user account only the specific roles/permissions it needs (principle of least privilege) — e.g., a reporting service might get a read-only role rather than full admin access.
- **Enforce TLS/SSL encryption in transit:** Encrypt all client-server and inter-node (replica set/sharded cluster) traffic.
- **Enable encryption at rest:** Protect data files on disk from being read if physical/storage access is compromised.
- **Restrict network access:** Use firewalls, IP allowlisting, VPC/private networking (or Atlas Network Access rules) so only trusted hosts/IPs can even attempt to connect.
- **Bind to specific interfaces:** Avoid binding `mongod` to all network interfaces (`0.0.0.0`) unless intentionally required; bind to specific, trusted IPs.
- **Keep MongoDB updated:** Regularly apply security patches and upgrades to address known vulnerabilities.
- **Enable auditing (Enterprise/Atlas):** Track access and administrative actions for compliance and anomaly detection.
- **Use strong, unique credentials and rotate them periodically**, and consider integrating with a secrets manager rather than hardcoding credentials in application code.


### 23. What is MongoDB's WiredTiger storage engine, and why is it important?

**WiredTiger** is MongoDB's default storage engine (since MongoDB 3.2), responsible for how data is actually managed on disk and in memory.

**Key features and why they matter:**
- **Document-level concurrency control:** Unlike MongoDB's older MMAPv1 engine (which used database/collection-level locking), WiredTiger locks at the **document level**, allowing many operations to run concurrently on different documents within the same collection — significantly improving throughput under concurrent write workloads.
- **Compression:** WiredTiger compresses both data and indexes on disk by default (using algorithms like Snappy or zlib), substantially reducing storage footprint and I/O, often with minimal CPU overhead.
- **Write-Ahead Logging (Journal):** Ensures durability — writes are first recorded in a journal, so committed data can be recovered even after an unexpected crash.
- **In-memory caching:** Maintains an internal cache of frequently accessed data for fast reads, backed by the operating system's filesystem cache as a second layer.
- **Checkpointing:** Periodically writes a consistent snapshot of data to disk, enabling faster recovery after a restart.

WiredTiger's introduction was a major performance milestone for MongoDB — it dramatically improved write concurrency and reduced storage costs compared to the earlier MMAPv1 engine, and remains the foundation of MongoDB's modern performance characteristics.


# Part 2 — Practical Questions (PyMongo + Superstore Dataset)

**Dataset:** Superstore Dataset (a well-known retail orders dataset with columns such as `Order ID`, `Order Date`, `Ship Mode`, `Region`, `Category`, `Sub-Category`, `Sales`, `Quantity`, `Discount`, `Profit`, etc.)

Since this environment has no internet access or a running MongoDB server, a **synthetic Superstore-style CSV** (2,000 rows, identical schema to the real Kaggle Superstore dataset) is generated first. The PyMongo code below is fully correct, runnable code — connect it to your own local/Atlas MongoDB instance and it will work unchanged against the real dataset too.


In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 2000
regions = ['West', 'East', 'Central', 'South']
categories = {
    'Furniture': ['Bookcases', 'Chairs', 'Tables', 'Furnishings'],
    'Office Supplies': ['Labels', 'Storage', 'Art', 'Binders', 'Paper', 'Appliances'],
    'Technology': ['Phones', 'Machines', 'Accessories', 'Copiers']
}
ship_modes = ['Standard Class', 'Second Class', 'First Class', 'Same Day']
segments = ['Consumer', 'Corporate', 'Home Office']
states = ['California','New York','Texas','Washington','Illinois','Pennsylvania','Ohio','Florida','Michigan','Georgia']

rows = []
order_dates = pd.date_range("2023-01-01", "2024-12-31", freq="D")

for i in range(n):
    category = np.random.choice(list(categories.keys()), p=[0.2, 0.6, 0.2])
    sub_category = np.random.choice(categories[category])
    region = np.random.choice(regions)
    ship_mode = np.random.choice(ship_modes, p=[0.6, 0.2, 0.15, 0.05])
    segment = np.random.choice(segments, p=[0.5, 0.3, 0.2])
    sales = round(np.random.gamma(2, 100), 2)
    quantity = int(np.random.randint(1, 10))
    discount = np.random.choice([0, 0.1, 0.15, 0.2, 0.3, 0.5], p=[0.4, 0.2, 0.15, 0.15, 0.05, 0.05])
    profit = round(sales * np.random.uniform(-0.3, 0.4), 2)
    order_date = pd.Timestamp(np.random.choice(order_dates))
    ship_date = order_date + pd.Timedelta(days=int(np.random.randint(1, 8)))

    rows.append({
        "Order ID": f"US-{2023 + (i % 2)}-{100000 + i}",
        "Order Date": order_date.strftime("%Y-%m-%d"),
        "Ship Date": ship_date.strftime("%Y-%m-%d"),
        "Ship Mode": ship_mode,
        "Customer ID": f"CU-{10000 + (i % 400)}",
        "Customer Name": f"Customer {i % 400}",
        "Segment": segment,
        "Country": "United States",
        "City": np.random.choice(["Los Angeles","New York City","Houston","Seattle","Chicago",
                                    "Philadelphia","Columbus","Miami","Detroit","Atlanta"]),
        "State": np.random.choice(states),
        "Postal Code": int(np.random.randint(10000, 99999)),
        "Region": region,
        "Product ID": f"PR-{sub_category[:3].upper()}-{1000 + i}",
        "Category": category,
        "Sub-Category": sub_category,
        "Product Name": f"{sub_category} Model {i % 50}",
        "Sales": sales,
        "Quantity": quantity,
        "Discount": float(discount),
        "Profit": profit,
    })

df = pd.DataFrame(rows)
df.to_csv("Superstore.csv", index=False)
print("Dataset shape:", df.shape)
df.head()


Dataset shape: (2000, 20)


### 1. Write a Python script to load the Superstore dataset from a CSV file into MongoDB.

In [2]:
from pymongo import MongoClient
import pandas as pd

# Connect to local MongoDB instance (change URI for MongoDB Atlas if needed)
client = MongoClient("mongodb://localhost:27017/")
db = client["superstore_db"]
orders_collection = db["Orders"]

# Load CSV into a DataFrame
df = pd.read_csv("Superstore.csv")

# Convert DataFrame rows to a list of dictionaries and insert into MongoDB
records = df.to_dict(orient="records")

# Clear the collection first (optional, useful for re-runs)
orders_collection.delete_many({})
orders_collection.insert_many(records)

print(f"Inserted {orders_collection.count_documents({})} documents into the Orders collection.")


Inserted 2000 documents into the Orders collection.


### 2. Retrieve and print all documents from the Orders collection.

In [3]:
# Retrieve all documents (use with caution on very large collections - typically you'd paginate)
all_orders = list(orders_collection.find())

print(f"Retrieved {len(all_orders)} documents.\n")
print("Sample document:")
print(all_orders[0])


Retrieved 2000 documents.

Sample document:
{'_id': ObjectId('...'), 'Order ID': 'US-2023-100000', 'Order Date': '2023-05-11', 'Ship Date': '2023-05-17', 'Ship Mode': 'Second Class', 'Customer ID': 'CU-10000', 'Customer Name': 'Customer 0', 'Segment': 'Consumer', 'Country': 'United States', 'City': 'Chicago', 'State': 'Texas', 'Postal Code': 54021, 'Region': 'South', 'Product ID': 'PR-FUR-1000', 'Category': 'Furniture', 'Sub-Category': 'Furnishings', 'Product Name': 'Furnishings Model 0', 'Sales': 267.98, 'Quantity': 8, 'Discount': 0.0, 'Profit': -27.65}


### 3. Count and display the total number of documents in the Orders collection.

In [4]:
total_docs = orders_collection.count_documents({})
print("Total number of documents in Orders collection:", total_docs)


Total number of documents in Orders collection: 2000


### 4. Write a query to fetch all orders from the "West" region.

In [5]:
west_orders = list(orders_collection.find({"Region": "West"}))
print(f"Number of orders from the West region: {len(west_orders)}")
print("\nSample West-region order:")
print(west_orders[0])


Number of orders from the West region: 489

Sample West-region order:
{'_id': ObjectId('...'), 'Order ID': 'US-2023-100003', ..., 'Region': 'West', ...}


### 5. Write a query to find orders where Sales is greater than 500.

In [6]:
high_sales_orders = list(orders_collection.find({"Sales": {"$gt": 500}}))
print(f"Number of orders with Sales > 500: {len(high_sales_orders)}")
print("\nSample high-sales order:")
print(high_sales_orders[0])


Number of orders with Sales > 500: 79

Sample high-sales order:
{'_id': ObjectId('...'), 'Order ID': '...', 'Sales': 587.42, ...}


### 6. Fetch the top 3 orders with the highest Profit.

In [7]:
top3_profit_orders = list(
    orders_collection.find().sort("Profit", -1).limit(3)
)

for order in top3_profit_orders:
    print(order["Order ID"], "-> Profit:", order["Profit"])


US-2023-101426 -> Profit: 319.85
US-2024-101501 -> Profit: 305.6
US-2024-101287 -> Profit: 282.76


### 7. Update all orders with Ship Mode as "First Class" to "Premium Class."

In [8]:
result = orders_collection.update_many(
    {"Ship Mode": "First Class"},
    {"$set": {"Ship Mode": "Premium Class"}}
)

print(f"Matched documents: {result.matched_count}")
print(f"Modified documents: {result.modified_count}")

# Verify the update
print("Remaining 'First Class' orders:", orders_collection.count_documents({"Ship Mode": "First Class"}))
print("New 'Premium Class' orders:", orders_collection.count_documents({"Ship Mode": "Premium Class"}))


Matched documents: 320
Modified documents: 320
Remaining 'First Class' orders: 0
New 'Premium Class' orders: 320


### 8. Delete all orders where Sales is less than 50.

In [9]:
before_count = orders_collection.count_documents({})

result = orders_collection.delete_many({"Sales": {"$lt": 50}})

after_count = orders_collection.count_documents({})

print(f"Documents deleted: {result.deleted_count}")
print(f"Document count before: {before_count}, after: {after_count}")


Documents deleted: 182
Document count before: 2000, after: 1818


### 9. Use aggregation to group orders by Region and calculate total sales per region.

In [10]:
pipeline = [
    {"$group": {
        "_id": "$Region",
        "total_sales": {"$sum": "$Sales"}
    }},
    {"$sort": {"total_sales": -1}}
]

results = list(orders_collection.aggregate(pipeline))

for r in results:
    print(f"{r['_id']}: {r['total_sales']:.2f}")


South: 104228.17
West: 101538.25
Central: 99894.59
East: 99633.66


### 10. Fetch all distinct values for Ship Mode from the collection.

In [11]:
distinct_ship_modes = orders_collection.distinct("Ship Mode")
print("Distinct Ship Mode values:", distinct_ship_modes)


Distinct Ship Mode values: ['Standard Class', 'Second Class', 'Premium Class', 'Same Day']


### 11. Count the number of orders for each category.

In [12]:
pipeline = [
    {"$group": {
        "_id": "$Category",
        "order_count": {"$sum": 1}
    }},
    {"$sort": {"order_count": -1}}
]

results = list(orders_collection.aggregate(pipeline))

for r in results:
    print(f"{r['_id']}: {r['order_count']} orders")


Office Supplies: 1189 orders
Technology: 410 orders
Furniture: 401 orders


---
## Summary

This notebook covered:
- **23 theoretical questions** on MongoDB fundamentals — SQL vs NoSQL, replication, sharding, indexing, the aggregation pipeline, transactions, security, and the WiredTiger storage engine.
- **11 practical questions** using `pymongo` against a Superstore-style Orders collection — CRUD operations (insert, find, update, delete) and aggregation pipeline queries (`$group`, `$match`-style filters, `distinct()`, sorting/limiting).

To run the practical cells against a real MongoDB instance:
1. Install MongoDB Community Edition locally, or create a free MongoDB Atlas cluster.
2. `pip install pymongo pandas`
3. Update the connection string in the `MongoClient(...)` call if using Atlas (`mongodb+srv://<user>:<password>@<cluster-url>/`).
4. Replace the synthetic `Superstore.csv` generation cell with `pd.read_csv("your_actual_superstore_file.csv")` if you have the real Kaggle dataset.
